# Lab 3A: RAG quality and grounding (10 minutes)

**Presenter: Nagarjuna Naidu** | Day 3 | about 10 minutes

- See how chunk size changes retrieval quality (Recall@3)
- See why hybrid search (keyword + vector) beats either alone
- Build a grounded answer with citations and a refusal rule

## 1. Setup: 48 documents, embeddings, cosine

In [ ]:
import os, re, glob, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from fastembed import TextEmbedding
from sklearn.feature_extraction.text import TfidfVectorizer
from llm_client import call_llm, llm_available, llm_mode
warnings.filterwarnings("ignore")
INK, TEAL, LIME, CORAL, GOLD, PURPLE = "#0B1F2A", "#0E7C86", "#9BD13B", "#E4572E", "#F2B134", "#6C4AB6"

DOCS = []
for f in sorted(glob.glob("data/corpus/*.txt")):
    head, body = open(f, encoding="utf-8").read().split("\n\n", 1)
    meta = dict(l.split(": ", 1) for l in head.splitlines() if ": " in l)
    DOCS.append(dict(id=os.path.basename(f).split("_")[0], title=meta["title"], date=meta.get("date", ""), text=body.strip()))
QUERIES = json.load(open("data/eval_queries.json"))
GOLD_Q = [q for q in QUERIES if q["type"] != "no_answer"]

_model = TextEmbedding("BAAI/bge-small-en-v1.5")
def embed(texts):
    """Normalised embedding vectors, shape (n, 384). Cosine = dot product."""
    v = np.array(list(_model.embed(list(texts))), dtype=float)
    return v / np.linalg.norm(v, axis=1, keepdims=True)

print(len(DOCS), "documents,", len(GOLD_Q), "gold queries (no_answer skipped)")

## 2. Chunking matters
Each chunk keeps its parent `doc_id`; a document's score is its best chunk. Vector search only.

In [ ]:
def make_chunks(n_words):
    out = []
    for d in DOCS:
        words = re.findall(r"\S+\s*", d["text"])
        step = n_words if n_words else max(len(words), 1)
        for i in range(0, len(words), step):
            out.append(dict(doc_id=d["id"], title=d["title"], text="".join(words[i:i + step]).strip()))
    return out

def vector_doc_ranking(qvec, chunks, cvecs):
    """Doc ids ranked by their best chunk's cosine similarity."""
    sims = cvecs @ qvec
    best = {}
    for c, s in zip(chunks, sims):
        best[c["doc_id"]] = max(best.get(c["doc_id"], -1), s)
    return sorted(best, key=best.get, reverse=True)

def recall_at_3(rank_fn, qtypes=("semantic", "keyword", "hybrid")):
    """Share of gold queries (per type) with at least one relevant doc in the top 3."""
    hits = {t: [] for t in qtypes}
    for q in GOLD_Q:
        hits[q["type"]].append(any(d in q["relevant_doc_ids"] for d in rank_fn(q["query"])[:3]))
    return {t: float(np.mean(h)) for t, h in hits.items()} | {"all": float(np.mean(sum(hits.values(), [])))}

QVEC = {q["query"]: v for q, v in zip(QUERIES, embed([q["query"] for q in QUERIES]))}
SETS = {"~60 words": 60, "~150 words": 150, "whole document": 0}
rows, CH = {}, {}
for name, n in SETS.items():
    ch = make_chunks(n); cv = embed([c["text"] for c in ch]); CH[name] = (ch, cv)
    rows[name] = {"chunks": len(ch)} | recall_at_3(lambda q, ch=ch, cv=cv: vector_doc_ranking(QVEC[q], ch, cv))
tbl = pd.DataFrame(rows).T
print(tbl.round(2).to_string())
ax = tbl["all"].plot.bar(color=[TEAL, CORAL, GOLD], figsize=(4, 2.2), rot=0); ax.set_ylim(0, 1); ax.set_ylabel("Recall@3"); plt.tight_layout(); plt.show()

**What to notice:** Recall@3 does not rise steadily with chunk size: ~60 words and whole documents tie at 0.90 overall while ~150 words trails at 0.86.
With only 29 queries (6 keyword), one query moves a row a lot, so treat chunk size as something you measure on your own data, not guess.

## 3. Hybrid beats either alone
Keyword = TF-IDF cosine on whole documents. Hybrid = Reciprocal Rank Fusion, score = sum of 1/(60 + rank).

In [ ]:
tfidf = TfidfVectorizer(sublinear_tf=True, token_pattern=r"[A-Za-z0-9][A-Za-z0-9\-]*").fit([d["text"] + " " + d["title"] for d in DOCS])
DM = tfidf.transform([d["text"] + " " + d["title"] for d in DOCS])
IDS = [d["id"] for d in DOCS]
CH150, CV150 = CH["~150 words"]

def keyword_rank(q):
    s = (DM @ tfidf.transform([q]).T).toarray().ravel()
    return [IDS[i] for i in np.argsort(-s)]
def vector_rank(q): return vector_doc_ranking(QVEC[q], CH150, CV150)
def hybrid_rank(q, k=60):
    score = {}
    for ranking in (keyword_rank(q), vector_rank(q)):
        for r, d in enumerate(ranking, 1):
            score[d] = score.get(d, 0) + 1 / (k + r)
    return sorted(score, key=score.get, reverse=True)

res = pd.DataFrame({"keyword": recall_at_3(keyword_rank), "vector": recall_at_3(vector_rank), "hybrid": recall_at_3(hybrid_rank)})
print(res.round(2).to_string())

**What to notice:** keyword search wins on exact codes (IDs), vector search on paraphrases; hybrid is the safest single choice overall (see the `all` row), at a small cost on `semantic` queries.

## 4. Grounded answers and refusal
Free LLM? Set three environment variables (same as Day 2): `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`.
Uncomment the cell below, paste your values, run it, then run the cell after it. Never share a notebook with a real key.
No key: the lab still runs in offline demo mode (quoted sentences, not generated text).

In [ ]:
# os.environ["LLM_API_KEY"]  = "PASTE-YOUR-KEY"
# os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"   # Google AI Studio: https://generativelanguage.googleapis.com/v1beta/openai/   Ollama: http://localhost:11434/v1
# os.environ["LLM_MODEL"]    = "PASTE-A-MODEL-NAME"

In [ ]:
print("LLM:", llm_mode())
SYSTEM = "Answer only from the context, cite sources like [S1], say you don't know otherwise."
REFUSAL = "I could not find this in the provided documents"
# Threshold chosen by looking at the gold queries themselves, so it is optimistic: it catches clearly
# off-topic questions, but our no_answer queries score as high as many real ones, so it will not catch them all.
THRESHOLD = 0.60

def answer(question, k=3):
    hits = sorted(zip(CV150 @ embed([question])[0], CH150), key=lambda x: -x[0])[:k]
    if hits[0][0] < THRESHOLD:
        return f"{REFUSAL}. (best similarity {hits[0][0]:.2f} < {THRESHOLD})", []
    context = "\n\n".join(f"[S{i}] {c['text']}" for i, (_, c) in enumerate(hits, 1))
    reply = call_llm(f"{context}\n\nQUESTION: {question}", system=SYSTEM)
    return reply, [f"[S{i}] {c['title']} ({c['doc_id']})" for i, (_, c) in enumerate(hits, 1)]

for q in ["can I work from Spain for a month", "how many weeks of leave does a primary carer get", "who won the 1998 world cup"]:
    a, src = answer(q)
    print(f"\nQ: {q}\nA: {a}")
    for s in src: print("   ", s)

**Parental leave:** the corpus holds both the 2022 and 2024 policy, and retrieval may cite either one. A freshness filter (prefer the newest `date`) is homework / Day 4.